In [1]:
# Import necessary libraries
import pandas as pd
import spacy
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim
from gensim.models import Word2Vec
import numpy as np

# For reproducibility, set a random seed if needed
import random
random.seed(42)
np.random.seed(42)

In [2]:
# Load your dataset
df = pd.read_csv(r"C:\Users\Edopi\Desktop\2024-25c-fai2-adsai-group-team_9\Data_lab_task\Task 3\STT_Assembly.csv", sep=',')  # Adjust separator if needed


In [3]:
# Load spaCy language model (change model as required by your language)
nlp = spacy.load('en_core_web_sm')

def extract_pos_tags(sentence):
    doc = nlp(sentence)
    # Extract the POS tag for each token in the sentence
    return [token.pos_ for token in doc]

df['POS_Tags'] = df['Sentence'].apply(extract_pos_tags)


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Sentence'])
# Convert each row of the sparse matrix to a dense list
df['TF_IDF'] = list(tfidf_matrix.toarray())


In [5]:
from textblob import TextBlob

def get_sentiment(sentence):
    blob = TextBlob(sentence)
    return blob.sentiment.polarity

df['Sentiment_Score'] = df['Sentence'].apply(get_sentiment)


In [6]:
from gensim.models import Word2Vec

# Prepare the sentences for training (split each sentence into words)
sentences = [sentence.split() for sentence in df['Sentence']]

# Train a custom Word2Vec model on your transcription data
custom_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

def average_custom_embedding(sentence):
    tokens = sentence.split()
    vectors = []
    for token in tokens:
        if token in custom_model.wv:
            vectors.append(custom_model.wv[token])
    if vectors:
        return np.mean(vectors, axis=0).tolist()
    else:
        return [0.0] * custom_model.vector_size

df['Custom_Embeddings'] = df['Sentence'].apply(average_custom_embedding)



In [7]:
def extract_named_entities(sentence):
    doc = nlp(sentence)
    # Return a list of tuples with the entity text and its label
    return [(ent.text, ent.label_) for ent in doc.ents]

df['NER_Entities'] = df['Sentence'].apply(extract_named_entities)


In [8]:
df.to_csv('NLP_features.tsv', sep=',', index=False)
